In [6]:
!pip install --upgrade backtesting

In [7]:
import yfinance as yf
import pandas as pd
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

# 1. Données préparées
data = yf.download('AAPL', period='2y')
data.columns = data.columns.get_level_values(0)

# 2. La fonction SMA
def SMA(valeurs, n):
    return pd.Series(valeurs).rolling(n).mean()

# 3. La stratégie
class CroisementSMA(Strategy):
    n_courte = 10
    n_longue = 20

    def init(self):
        prix = self.data.Close
        self.sma_courte = self.I(SMA, prix, self.n_courte)
        self.sma_longue = self.I(SMA, prix, self.n_longue)

    def next(self):
        if crossover(self.sma_courte, self.sma_longue):
            self.buy()
        elif crossover(self.sma_longue, self.sma_courte):
            self.position.close()

# 4. Lancer
bt = Backtest(data, CroisementSMA, cash=10000, commission=0)
resultats = bt.run()
print(resultats)

[*********************100%***********************]  1 of 1 completed


Backtest.run:   0%|          | 0/481 [00:00<?, ?bar/s]

Start                     2024-08-08 00:00:00
End                       2026-08-07 00:00:00
Duration                    729 days 00:00:00
Exposure Time [%]                    59.48104
Equity Final [$]                  11023.57958
Equity Peak [$]                   11924.47924
Return [%]                            10.2358
Buy & Hold Return [%]                41.95467
Return (Ann.) [%]                     5.03417
Volatility (Ann.) [%]                19.95181
CAGR [%]                              5.00376
Sharpe Ratio                          0.25232
Sortino Ratio                         0.36296
Calmar Ratio                          0.21463
Alpha [%]                            -8.05485
Beta                                  0.43596
Max. Drawdown [%]                    -23.4553
Avg. Drawdown [%]                     -4.8525
Max. Drawdown Duration      503 days 00:00:00
Avg. Drawdown Duration       51 days 00:00:00
# Trades                                   12
Win Rate [%]                      

In [10]:
# 1. Couper en deux (chronologiquement !)
n = len(data)
point_coupure = int(n * 0.7)        # 70% pour le train

data_train = data.iloc[:point_coupure]    # la partie ANCIENNE
data_test  = data.iloc[point_coupure:]    # la partie RÉCENTE

# Backtest sur le TRAIN
bt_train = Backtest(data_train, CroisementSMA, cash=10000, commission=0)

# Optimiser les paramètres SUR LE TRAIN
combinaisons = [(5, 20), (10, 30), (15, 50), (20, 60), (25, 80)]

meilleur_return = -999
meilleurs_params = None

for c, l in combinaisons:
    bt_train = Backtest(data_train, CroisementSMA, cash=10000, commission=0)
    res = bt_train.run(n_courte=c, n_longue=l)
    ret = res['Return [%]']
    print(f"courte={c}, longue={l} -> Return train = {ret:.2f}%")
    if ret > meilleur_return:
        meilleur_return = ret
        meilleurs_params = (c, l)

print()
print("Meilleure combinaison sur le TRAIN :", meilleurs_params, "->", round(meilleur_return, 2), "%")
print()

#  VERDICT SUR LE TEST 
meilleure_courte, meilleure_longue = meilleurs_params

bt_test = Backtest(data_test, CroisementSMA, cash=10000, commission=0)
res_test = bt_test.run(n_courte=meilleure_courte, n_longue=meilleure_longue)

print("VERDICT")
print("Return sur le TRAIN :", round(meilleur_return, 2), "%")
print("Return sur le TEST  :", round(res_test['Return [%]'], 2), "%")
print("Buy & Hold sur le TEST :", round(res_test['Buy & Hold Return [%]'], 2), "%")

Backtest.run:   0%|          | 0/330 [00:00<?, ?bar/s]

courte=5, longue=20 -> Return train = -10.70%


Backtest.run:   0%|          | 0/320 [00:00<?, ?bar/s]

courte=10, longue=30 -> Return train = 4.99%


Backtest.run:   0%|          | 0/300 [00:00<?, ?bar/s]

...

courte=25, longue=80 -> Return train = 28.31%

Meilleure combinaison sur le TRAIN : (25, 80) -> 28.31 %



C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_13268\3611109127.py:19: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  res = bt_train.run(n_courte=c, n_longue=l)


Backtest.run:   0%|          | 0/71 [00:00<?, ?bar/s]

=== VERDICT ===
Return sur le TRAIN : 28.31 %
Return sur le TEST  : 15.94 %
Buy & Hold sur le TEST : 17.19 %


C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_13268\3611109127.py:34: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  res_test = bt_test.run(n_courte=meilleure_courte, n_longue=meilleure_longue)
